In [ ]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/annotated_cpu/checkpoints/post_cell_42.pickle

In [ ]:
%%RecordEvent
%%time
### cell 43 ###

# vectorized percentage computation using value_counts(normalize=True)
def count_then_return_percent_55(dataframe, x_axis_title):
    return (
        dataframe[x_axis_title]
        .value_counts(dropna=False, normalize=True)
        .mul(100)
        .round(1)
    )

# combine across years in one loop, preserving index as the category label
def combine_subset_of_data_from_multiple_years_55(question_of_interest, x_axis_title, include_2017=None):
    year_sources = [
        ('2022', responses_df_2022_cell10),
        ('2021', responses_df_2021),
        ('2020', responses_df_2020),
        ('2019', responses_df_2019_cell10),
        ('2018', responses_df_2018_cell10),
    ]
    if include_2017 is not None:
        year_sources.append(('2017', responses_df_2017))
    # reverse to get chronological order: 2017→2022 or 2018→2022
    year_sources = list(reversed(year_sources))

    frames = []
    for year, df in year_sources:
        tmp = (
            count_then_return_percent_55(df, question_of_interest)
            .sort_index()
            .to_frame(name='percentage')
        )
        tmp['year'] = year
        frames.append(tmp)

    df_combined = pd.concat(frames)
    # bring the category (index) into its own column
    df_combined[x_axis_title] = df_combined.index
    # keep the same column order as the original
    return df_combined[['percentage', 'year', x_axis_title]]

question_of_interest_cell55 = (
    'Select the title most similar to your current role '
    '(or most recent title if retired): - Selected Choice'
)
job_df_combined = combine_subset_of_data_from_multiple_years_55(
    question_of_interest_cell55,
    title_for_x_axis_cell43
)
# replace the long label in the category column
job_df_combined.replace(
    'Data Analyst (Business, Marketing, Financial, Quantitative, etc)',
    'Data Analyst',
    inplace=True
)
# filter for the roles we care about (the index still holds the category label)
roles = ['Data Analyst', 'Data Engineer', 'Data Scientist', 'Research Scientist', 'Software Engineer']
job_df_combined_cell55 = job_df_combined[job_df_combined.index.isin(roles)]
job_df_combined_cell55.info()

In [ ]:
%Checkpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_43_try_1.pickle

In [ ]:
%PrintCellInfo opt_cell_exec_info

In [ ]:

with open("/home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/opt_cell_exec_info_43_try_1.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[43], f)


In [ ]:
opt_output = Out.get(4)